In [2]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split

In [3]:
import sys
import tensorflow as tf

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)

Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
TensorFlow: 2.21.0


In [7]:
plantvillage_path = r"D:\FruitDiseaseDissertation\Dataset\PlantVillage-Dataset-master\raw\color"

valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")

records = []  

for class_name in os.listdir(plantvillage_path):
    class_path = os.path.join(plantvillage_path, class_name)

    if os.path.isdir(class_path):
        for file_name in os.listdir(class_path):
            if file_name.lower().endswith(valid_extensions):
                records.append({
                    "image_path": os.path.join(class_path, file_name),
                    "class_name": class_name
                })

pv_df = pd.DataFrame(records)

print("images :",len(pv_df))
print("classes :",pv_df['class_name'].nunique())

images : 54305
classes : 38


In [8]:
# encodeing the lable 

class_names = sorted(pv_df["class_name"].unique())
class_to_index ={
    name:idx for idx,name in enumerate(class_names)
}

pv_df["label"] = pv_df["class_name"].map(class_to_index)

In [10]:
#spliting it to 70/15/15 train, validation and test sets

train_val_df, test_df = train_test_split(
    pv_df,
    test_size = 0.15,
    random_state = 42,
    stratify = pv_df["label"]
)

train_df,val_df = train_test_split(
    train_val_df,
    test_size = 0.1765,
    random_state = 42,
    stratify = train_val_df["label"]
)

In [11]:
print(len(train_df),len(val_df),len(test_df))

38011 8148 8146


In [12]:
from PIL import Image

bad_images = []

for path in pv_df["image_path"]:
    try:
        with Image.open(path) as img:
            img.verify()
    except Exception:
        bad_images.append(path)

print("Corrupt/unreadable images:", len(bad_images))

Corrupt/unreadable images: 0


In [13]:
IMG_SIZE = 224

def load_image(path,label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image,channels=3)
    image = tf.image.resize(image,[IMG_SIZE,IMG_SIZE])
    image = tf.cast(image,tf.float32)

    return image,label

In [17]:
BATH_size = 16

def make_dataset(df,training = False):
    paths = df["image_path"].values
    labels = df["label"].values

    ds = tf.data.Dataset.from_tensor_slices((paths,labels))

    if training:
        ds = ds.shuffle(
            buffer_size =len(df),
            seed=42
        )

    ds =ds.map(
        load_image,
        num_parallel_calls = tf.data.AUTOTUNE
    )

    ds = ds.batch(BATH_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

In [18]:
train_ds = make_dataset(train_df,training = True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

In [19]:
for images,labels in train_ds.take(1):
    print("Images:",images.shape)
    print("Labels:",labels.shape)
    print("Min pixel:",tf.reduce_min(images).numpy())
    print("Max pixel", tf.reduce_max(images).numpy())

Images: (16, 224, 224, 3)
Labels: (16,)
Min pixel: 0.0
Max pixel 255.0


In [21]:
import os

os.makedirs("D:\FruitDiseaseDissertation/processed_data",exist_ok = True)

train_df.to_csv("D:\FruitDiseaseDissertation/processed_data/pv_train.csv",index = False)
val_df.to_csv("D:\FruitDiseaseDissertation/processed_data/pv_val.csv", index = False)
test_df.to_csv("D:\FruitDiseaseDissertation/processed_data/pv_test.csv",index = False)

<>:3: SyntaxWarning: invalid escape sequence '\F'
<>:5: SyntaxWarning: invalid escape sequence '\F'
<>:6: SyntaxWarning: invalid escape sequence '\F'
<>:7: SyntaxWarning: invalid escape sequence '\F'
<>:3: SyntaxWarning: invalid escape sequence '\F'
<>:5: SyntaxWarning: invalid escape sequence '\F'
<>:6: SyntaxWarning: invalid escape sequence '\F'
<>:7: SyntaxWarning: invalid escape sequence '\F'
C:\Users\DELL\AppData\Local\Temp\ipykernel_19956\3513397653.py:3: SyntaxWarning: invalid escape sequence '\F'
  os.makedirs("D:\FruitDiseaseDissertation/processed_data",exist_ok = True)
C:\Users\DELL\AppData\Local\Temp\ipykernel_19956\3513397653.py:5: SyntaxWarning: invalid escape sequence '\F'
  train_df.to_csv("D:\FruitDiseaseDissertation/processed_data/pv_train.csv",index = False)
C:\Users\DELL\AppData\Local\Temp\ipykernel_19956\3513397653.py:6: SyntaxWarning: invalid escape sequence '\F'
  val_df.to_csv("D:\FruitDiseaseDissertation/processed_data/pv_val.csv", index = False)
C:\Users\DELL\A